### EXERCISE 6 TEXT SIMILARITY BY ALENA(254) & RENATA(093)

Menerapkan *text similarity* (Cosine dan Jaccard) pada teks hasil feature engineering dari Exercise 5.

#### Data & preprocessing

In [1]:
import pandas as pd, numpy as np
from IPython.display import display

texts = ['Cuti Bersama Idul Fitri', 'Cuti Bersama Idul Fitri', 'Cuti Bersama Idul Fitri', 'Cuti Bersama Idul Fitri',
         'Hari Idul Fitri', 'Cuti Bersama Idul Fitri', 'Idul Adha (Lebaran Haji)', 'Cuti Bersama Idul Fitri',
         'Idul Adha (Lebaran Haji)', 'Cuti Bersama Idul Fitri', 'Hari Idul Fitri', 'Cuti Bersama Idul Fitri',
         'Hari Idul Fitri', 'Cuti Bersama Idul Fitri', 'Hari Idul Fitri']

hasil_preprocess = {
    'Cuti Bersama Idul Fitri':  ['cuti', 'idul', 'fitri'],
    'Hari Idul Fitri':          ['idul', 'fitri'],
    'Idul Adha (Lebaran Haji)': ['idul', 'adha', 'lebaran', 'haji'],
}
df_pre = pd.Series([' '.join(hasil_preprocess[t]) for t in texts])
df_pre.value_counts()

cuti idul fitri           9
idul fitri                4
idul adha lebaran haji    2
Name: count, dtype: int64

#### 1. Cosine Similarity: CountVectorizer

In [2]:
from sklearn.feature_extraction.text import CountVectorizer, TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity
from scipy.spatial.distance import pdist, squareform, cdist

labels    = ['Cuti Bersama Idul Fitri', 'Hari Idul Fitri', 'Idul Adha (Lebaran Haji)']
idx       = [texts.index(l) for l in labels]  

def tampil(S, judul):
    print(judul)
    display(pd.DataFrame(S, index=labels, columns=labels).round(3))

X_cv = CountVectorizer().fit_transform(df_pre)[idx]
sim_cv = cosine_similarity(X_cv)
tampil(sim_cv, 'Cosine - CountVectorizer')

Cosine - CountVectorizer


,Cuti Bersama Idul Fitri,Hari Idul Fitri,Idul Adha (Lebaran Haji)
Cuti Bersama Idul Fitri,1.000,0.816,0.289
Hari Idul Fitri,0.816,1.000,0.354
Idul Adha (Lebaran Haji),0.289,0.354,1.000


#### 2. Cosine Similarity: TF-IDF

In [3]:
X_tf = TfidfVectorizer().fit_transform(df_pre)[idx]
sim_tf = cosine_similarity(X_tf)
tampil(sim_tf, 'Cosine - TF-IDF')

Cosine - TF-IDF


,Cuti Bersama Idul Fitri,Hari Idul Fitri,Idul Adha (Lebaran Haji)
Cuti Bersama Idul Fitri,1.000,0.717,0.10
Hari Idul Fitri,0.717,1.000,0.14
Idul Adha (Lebaran Haji),0.100,0.140,1.00


#### 3. Jaccard Similarity: binary

In [4]:
X_bin = CountVectorizer(binary=True).fit_transform(df_pre).toarray()[idx]
sim_jac = 1 - squareform(pdist(X_bin, metric='jaccard'))   
tampil(sim_jac, 'Jaccard')

Jaccard


,Cuti Bersama Idul Fitri,Hari Idul Fitri,Idul Adha (Lebaran Haji)
Cuti Bersama Idul Fitri,1.000,0.667,0.167
Hari Idul Fitri,0.667,1.000,0.200
Idul Adha (Lebaran Haji),0.167,0.200,1.000


#### 4. Ringkasan perbandingan

In [5]:
pairs = [(0, 1), (0, 2), (1, 2)]
ringkasan = pd.DataFrame({
    'Pasangan':        [f'{labels[i]}  vs  {labels[j]}' for i, j in pairs],
    'Cosine (Count)':  [sim_cv[i, j]  for i, j in pairs],
    'Cosine (TF-IDF)': [sim_tf[i, j]  for i, j in pairs],
    'Jaccard':         [sim_jac[i, j] for i, j in pairs],
}).round(3)
ringkasan

,Pasangan,Cosine (Count),Cosine (TF-IDF),Jaccard
0,Cuti Bersama Idul Fitri vs Hari Idul Fitri,0.816,0.717,0.667
1,Cuti Bersama Idul Fitri vs Idul Adha (Lebara...,0.289,0.100,0.167
2,Hari Idul Fitri vs Idul Adha (Lebaran Haji),0.354,0.140,0.200


#### 5. Pencarian dengan query (Top-K)

In [6]:
corpus = df_pre.tolist()
TopK = 3

tfidf = TfidfVectorizer()
X_tfidf = tfidf.fit_transform(corpus)

cv_bin = CountVectorizer(binary=True)
X_binary = cv_bin.fit_transform(corpus).toarray()

def cari(query, TopK=3):
    sim_t = cosine_similarity(X_tfidf, tfidf.transform([query]))[:, 0]
    sim_j = 1 - cdist(X_binary, cv_bin.transform([query]).toarray(), metric='jaccard')[:, 0]
    hasil = pd.DataFrame({'Teks': texts, 'TF-IDF Cosine': sim_t, 'Jaccard': sim_j})
    hasil = hasil.drop_duplicates('Teks').sort_values('TF-IDF Cosine', ascending=False).head(TopK)
    return hasil.reset_index(drop=True).round(3)

for q in ['lebaran haji', 'idul', 'cuti idul fitri']:
    print('Query:', q)
    display(cari(q))

Query: lebaran haji


,Teks,TF-IDF Cosine,Jaccard
0,Idul Adha (Lebaran Haji),0.798,0.5
1,Cuti Bersama Idul Fitri,0.000,0.0
2,Hari Idul Fitri,0.000,0.0


Query: idul


,Teks,TF-IDF Cosine,Jaccard
0,Hari Idul Fitri,0.662,0.500
1,Cuti Bersama Idul Fitri,0.474,0.333
2,Idul Adha (Lebaran Haji),0.211,0.250


Query: cuti idul fitri


,Teks,TF-IDF Cosine,Jaccard
0,Cuti Bersama Idul Fitri,1.000,1.000
1,Hari Idul Fitri,0.717,0.667
2,Idul Adha (Lebaran Haji),0.100,0.167
